## Pré-processamento e transformação dos dados


1. Importa os dados e verifica estrutura das tabelas

In [42]:
import pandas as pd

dados_labled = pd.read_csv('AgeOfMythology_work.csv')

dados_labled.info()

<class 'pandas.DataFrame'>
RangeIndex: 83 entries, 0 to 82
Data columns (total 18 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unit         83 non-null     str    
 1   Age          83 non-null     str    
 2   Cost         83 non-null     str    
 3   Pop          83 non-null     str    
 4   DmgHack      83 non-null     str    
 5   DmgPierce    83 non-null     str    
 6   DmgCrush     83 non-null     str    
 7   Range        83 non-null     str    
 8   HP           83 non-null     str    
 9   ArmorHack    83 non-null     str    
 10  ArmorPierce  83 non-null     str    
 11  ArmorCrush   83 non-null     str    
 12  Speed        83 non-null     float64
 13  LOS          83 non-null     int64  
 14  Train time   83 non-null     str    
 15  Bonus        66 non-null     str    
 16  Other        10 non-null     str    
 17  Class        83 non-null     str    
dtypes: float64(1), int64(1), str(16)
memory usage: 11.8 KB


In [43]:
dados_unlabled = pd.read_csv('AgeOfMythology_noClass.csv')

dados_unlabled.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Age          10 non-null     int64  
 1   Pop          10 non-null     int64  
 2   DmgHack      10 non-null     int64  
 3   DmgPierce    10 non-null     int64  
 4   DmgCrush     10 non-null     int64  
 5   Range        10 non-null     int64  
 6   HP           10 non-null     int64  
 7   ArmorHack    10 non-null     int64  
 8   ArmorPierce  10 non-null     int64  
 9   ArmorCrush   10 non-null     int64  
 10  Speed        10 non-null     float64
 11  Train time   10 non-null     int64  
 12  Favor        10 non-null     int64  
dtypes: float64(1), int64(12)
memory usage: 1.1 KB


2. Verifica quais colunas presentes nos dados rotulados estão ausentes nos não rotulados.

obs: Explicações do professor sobre as colunas:
- LOS: line of sights
- Bonus: contra quem ela é forte (negativo é fraco)
- Other: atributo especial
- Favor: outro recurso (custo de criação de unidade)

In [44]:
for column in dados_labled.columns:
    if column not in dados_unlabled.columns and column != 'Class':
        print(column)

Unit
Cost
LOS
Bonus
Other


Com a exceção de "Cost", vamos remover essas colunas pois não serão úteis para classificação. _"Cost"_ se refere ao custo de criação ed uma unidade, o qual pode fazer uso de diversos recursos diferentes. Um desses recursos é _"Favor"_ o qual está presente na tabela de dados não rotulados e será útil na classificação.

In [45]:
dados_labled['Cost'].head(7)

0      50F 2Fv
1    200W 15Fv
2    250F 22Fv
3    200F 16Fv
4    250F 28Fv
5    300W 28Fv
6    200G 25Fv
Name: Cost, dtype: str

3. Cria uma nova coluna Favor com os números seguidos por "Fv" da coluna Cost e remove a coluna Cost

In [ ]:
dados_labled['Favor'] = (
    dados_labled['Cost'].str.extract(r'(\d+(?:\.\d+)?)\s*Fv', expand=False)
    .astype(float).fillna(0).round().astype('Int64')
)
dados_labled = dados_labled.drop(columns=['Cost'])

dados_labled[['Unit', 'Favor']].head(7)

In [46]:
colunas_para_remover = dados_labled.columns.difference(dados_unlabled.columns).difference(["Class"])
dados_labled = dados_labled.drop(columns=colunas_para_remover)

dados_labled.columns

Index(['Age', 'Pop', 'DmgHack', 'DmgPierce', 'DmgCrush', 'Range', 'HP',
       'ArmorHack', 'ArmorPierce', 'ArmorCrush', 'Speed', 'Train time',
       'Class'],
      dtype='str')

Percebe-se algumas colunas do dataset rotulado estão com o tipo de dado incorreto (colunas str eram pra ser Int64). Conversão dos tipos de dados:

In [47]:
for coluna, serie in dados_unlabled.items():
    if coluna in dados_labled.columns and pd.api.types.is_integer_dtype(serie):
        coluna_convertida = pd.to_numeric(dados_labled[coluna], errors="coerce")
        dados_labled[coluna] = coluna_convertida.round().astype("Int64")

dados_labled.head()

,Age,Pop,DmgHack,DmgPierce,DmgCrush,Range,HP,ArmorHack,ArmorPierce,ArmorCrush,Speed,Train time,Class
0,1,2,<NA>,<NA>,<NA>,0,140,50,50,99,5.0,6,Myth Unit
1,2,3,<NA>,12,<NA>,12,220,25,35,80,5.0,19,Myth Unit
2,2,4,15,<NA>,12,2,600,40,50,80,3.2,20,Myth Unit
3,2,4,15,<NA>,10,2,300,60,50,80,4.0,20,Myth Unit
4,3,5,20,<NA>,10,3,800,60,40,80,4.0,20,Myth Unit


TO DO: pegar favor do cost.
TO DO: age -- == 0